In [1]:
from dotenv import load_dotenv

load_dotenv()


True

In [3]:
from langchain.agents import create_agent
from sidekick_tools import search, send_push_notification, wikipedia_lookup
from langgraph.checkpoint.memory import InMemorySaver

simple_worker = create_agent(
    model="openrouter:cohere/north-mini-code:free",
    tools=[search, send_push_notification, wikipedia_lookup],
    system_prompt="You are Sidekick, a helpful personal assistant. Use your tools to complete the task.",
    checkpointer=InMemorySaver(),
)

In [4]:
async def ask(worker, message):
    config = {"configurable": {"thread_id": "simple-sidekick"}}
    result = await worker.ainvoke({"messages": [{"role": "user", "content": message}]}, config=config)
    return result["messages"][-1].content

reply = await ask(simple_worker, "Search for who won the Nobel Prize in Physics in 2023 and send a push notification with a short summary.")
print(reply)

I've found the information about the 2023 Nobel Prize in Physics winners and sent you a push notification with the summary.

The 2023 Nobel Prize in Physics was awarded to **Pierre Agostini, Anne L'Huillier, and Ferenc Krausz** for their work on developing experimental methods to generate attosecond laser pulses. These ultra-fast pulses, with durations of one billionth of a billionth of a second (10^-18 seconds), enable scientists to study electron dynamics in matter and observe how electrons rearrange themselves at extremely rapid timescales.

The push notification has been sent to your phone with this information!


In [5]:
# Human in the loop:

from langchain_core.tools import tool


@tool
def book_meeting(person: str, day: str) -> str:
    """Book a meeting with a person on a given day."""
    return f"meeting booked with {person} on {day}"

In [11]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

approval_agent = create_agent(
    model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free",
    tools=[book_meeting],
    system_prompt="You are a scheduling assistant. Use the book_meeting tool.",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"book_meeting": True})],
    checkpointer=InMemorySaver()
)

config = {
    "configurable": {
        "thread_id": 'approval_demo'
    }
}

result = await approval_agent.ainvoke(
    {"messages": [{"role": "user", "content": "Book a meeting with Sam on Friday."}]}, config
)

interrupt = result["__interrupt__"][0]
print("The agent paused and is asking for approval:")
print(interrupt.value["action_requests"][0]["description"])

The agent paused and is asking for approval:
Tool execution requires approval

Tool: book_meeting
Args: {'person': 'Sam', 'day': 'Friday'}


In [12]:
from langgraph.types import Command


resumed = await approval_agent.ainvoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
print(resumed['messages'][-1].content)

I've successfully booked a meeting with Sam for Friday.


In [13]:
reject_config = {
    "configurable": {
        "thread_id": "reject"
    }
}

result = await approval_agent.ainvoke(
    {"messages": [{"role": "user", "content": "Book a meeting with Sam on Friday."}]}, reject_config
)

interrupt = result["__interrupt__"][0]
print("agent paused:")
print(interrupt.value["action_requests"][0]['description'])

agent paused:
Tool execution requires approval

Tool: book_meeting
Args: {'day': 'Friday', 'person': 'Sam'}


In [14]:
reject = await approval_agent.ainvoke(Command(resume={"decisions": [{"type": "reject"}]}), config=reject_config)
print(reject['messages'][-1].content)

I attempted to book a meeting with Sam on Friday, but the request was not completed. Would you like me to try again or do you need to provide any additional details about the meeting?
